# 数据清洗

本Notebook完成以下数据清洗工作：
1. 单表清洗（6步流程）
2. 宽表与长表转换
3. 多表合并

每步均有文字说明和前后对比。

## 1. 环境准备与数据加载

In [1]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 股票信息
stock_list = {
    '603063': '禾望电气',
    '002518': '科士达',
    '603118': '共进股份',
    '000063': '中兴通讯',
    '600036': '招商银行',
    '002142': '宁波银行',
    '002594': '比亚迪',
    '000002': '万科A',
    '600519': '贵州茅台',
    '002352': '顺丰控股'
}

stock_industry = {
    '603063': '能源',
    '002518': '能源',
    '603118': '通讯',
    '000063': '通讯',
    '600036': '银行',
    '002142': '银行',
    '002594': '汽车',
    '000002': '房地产',
    '600519': '白酒',
    '002352': '物流'
}

In [2]:
# 加载所有股票数据（真实数据 - AKShare新浪数据源）
stock_data = {}
for code, name in stock_list.items():
    file_path = f'data/stock/{code}.csv'
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['code'] = code
        df['name'] = name
        df['industry'] = stock_industry[code]
        stock_data[code] = df
        print(f"{name}({code}): {df.shape}")
    else:
        print(f"⚠️ {name}({code}): 文件不存在")

# 加载指数数据
hs300 = pd.read_csv('data/index/hs300.csv')
hs300['code'] = '000300'
hs300['name'] = '沪深300'
print(f"\n沪深300: {hs300.shape}")

cyb = pd.read_csv('data/index/cyb.csv')
cyb['code'] = '399006'
cyb['name'] = '创业板指'
print(f"创业板指: {cyb.shape}")

# 加载宏观数据
cpi = pd.read_csv('data/macro/cpi.csv')
print(f"\nCPI: {cpi.shape}")

m2 = pd.read_csv('data/macro/m2.csv')
print(f"M2: {m2.shape}")

# 加载财务数据
finance = pd.read_csv('data/finance/finance_indicators.csv')
print(f"\n财务指标: {finance.shape}")

禾望电气(603063): (1540, 10)
科士达(002518): (1540, 10)
共进股份(603118): (1535, 10)
中兴通讯(000063): (1539, 10)
招商银行(600036): (1540, 10)
宁波银行(002142): (1534, 10)
比亚迪(002594): (1540, 10)
万科A(000002): (1540, 10)
贵州茅台(600519): (1540, 10)
顺丰控股(002352): (1537, 10)

沪深300: (1540, 8)
创业板指: (1540, 8)

CPI: (70, 2)
M2: (130, 2)

财务指标: (40, 4)


## 2. 单表清洗（6步流程）

对每只股票数据依次执行6步清洗，并汇总结果。

### 2.1 步骤1：缺失值检测

统计每只股票各列的缺失值数量和比例，并分析可能原因。

In [3]:
print("=" * 70)
print("步骤1：缺失值检测")
print("=" * 70)

missing_summary = []
for code, df in stock_data.items():
    for col in ['date', 'open', 'close', 'high', 'low', 'volume', 'amount']:
        missing_count = df[col].isna().sum()
        missing_pct = missing_count / len(df) * 100
        missing_summary.append({
            'code': code,
            'name': stock_list[code],
            'column': col,
            'missing_count': missing_count,
            'missing_pct': round(missing_pct, 2)
        })

df_missing = pd.DataFrame(missing_summary)
df_missing_nonzero = df_missing[df_missing['missing_count'] > 0]

if len(df_missing_nonzero) > 0:
    print("发现缺失值：")
    print(df_missing_nonzero.to_string(index=False))
else:
    print("✅ 所有股票数据均无缺失值")

print("\n可能原因分析：")
print("- A股日度交易数据通常不缺失，因为交易所每个交易日都有完整行情")
print("- 极端情况下，股票停牌期间不会有交易数据，但停牌本身不在时间序列中产生缺失")
print("- 如果存在缺失，最可能的原因是数据源在特定日期未采集到数据")

步骤1：缺失值检测
✅ 所有股票数据均无缺失值

可能原因分析：
- A股日度交易数据通常不缺失，因为交易所每个交易日都有完整行情
- 极端情况下，股票停牌期间不会有交易数据，但停牌本身不在时间序列中产生缺失
- 如果存在缺失，最可能的原因是数据源在特定日期未采集到数据


### 2.2 步骤2：缺失值处理

**处理策略**：对于时间序列数据，缺失值采用**向前填充**（forward fill），因为：
- 股票行情是时间序列数据，相邻交易日数据相关性高
- 向前填充可以保持序列的连续性，避免引入虚假波动
- 删除缺失值会破坏时间序列的完整性

In [4]:
print("=" * 70)
print("步骤2：缺失值处理")
print("=" * 70)

for code, df in stock_data.items():
    missing_before = df.isna().sum().sum()
    
    # 向前填充
    df.fillna(method='ffill', inplace=True)
    # 如果开头仍有缺失，向后填充
    df.fillna(method='bfill', inplace=True)
    
    missing_after = df.isna().sum().sum()
    stock_data[code] = df
    
    if missing_before > 0:
        print(f"{stock_list[code]}({code}): 填充 {missing_before} 个缺失值")

print("\n✅ 缺失值处理完成")

步骤2：缺失值处理

✅ 缺失值处理完成


### 2.3 步骤3：日期格式统一

将日期列统一为 `datetime64` 格式，并设为索引。

In [5]:
print("=" * 70)
print("步骤3：日期格式统一")
print("=" * 70)

for code, df in stock_data.items():
    # 转换日期格式
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    stock_data[code] = df
    
    print(f"{stock_list[code]}({code}): 日期范围 {df.index.min().strftime('%Y-%m-%d')} ~ {df.index.max().strftime('%Y-%m-%d')}")

# 同样处理指数数据
hs300['date'] = pd.to_datetime(hs300['date'])
hs300 = hs300.set_index('date').sort_index()

cyb['date'] = pd.to_datetime(cyb['date'])
cyb = cyb.set_index('date').sort_index()

print("\n✅ 日期格式统一完成")

步骤3：日期格式统一
禾望电气(603063): 日期范围 2020-01-02 ~ 2026-05-15
科士达(002518): 日期范围 2020-01-02 ~ 2026-05-15
共进股份(603118): 日期范围 2020-01-02 ~ 2026-05-15
中兴通讯(000063): 日期范围 2020-01-02 ~ 2026-05-15
招商银行(600036): 日期范围 2020-01-02 ~ 2026-05-15
宁波银行(002142): 日期范围 2020-01-02 ~ 2026-05-15
比亚迪(002594): 日期范围 2020-01-02 ~ 2026-05-15
万科A(000002): 日期范围 2020-01-02 ~ 2026-05-15
贵州茅台(600519): 日期范围 2020-01-02 ~ 2026-05-15
顺丰控股(002352): 日期范围 2020-01-02 ~ 2026-05-15

✅ 日期格式统一完成


### 2.4 步骤4：数据类型检查

确认数值型字段为 `float64` 或 `int64`，字符型字段已正确标注。

In [6]:
print("=" * 70)
print("步骤4：数据类型检查")
print("=" * 70)

for code, df in stock_data.items():
    # 确保数值列为数值类型
    for col in ['open', 'close', 'high', 'low', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    stock_data[code] = df
    print(f"{stock_list[code]}({code}):")
    print(f"  数据类型: {df[['open', 'close', 'high', 'low', 'volume', 'amount']].dtypes.to_dict()}")

print("\n✅ 数据类型检查完成")

步骤4：数据类型检查
禾望电气(603063):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'volume': dtype('float64'), 'amount': dtype('float64')}
科士达(002518):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'volume': dtype('float64'), 'amount': dtype('float64')}
共进股份(603118):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'volume': dtype('float64'), 'amount': dtype('float64')}
中兴通讯(000063):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'volume': dtype('float64'), 'amount': dtype('float64')}
招商银行(600036):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'volume': dtype('float64'), 'amount': dtype('float64')}
宁波银行(002142):
  数据类型: {'open': dtype('float64'), 'close': dtype('float64'),

### 2.5 步骤5：重复值处理

检测并删除重复的日期索引记录。

In [7]:
print("=" * 70)
print("步骤5：重复值处理")
print("=" * 70)

total_duplicates = 0
for code, df in stock_data.items():
    dup_count = df.index.duplicated().sum()
    if dup_count > 0:
        df = df[~df.index.duplicated(keep='first')]
        stock_data[code] = df
        print(f"{stock_list[code]}({code}): 删除 {dup_count} 条重复记录")
        total_duplicates += dup_count

if total_duplicates == 0:
    print("✅ 所有股票数据均无重复值")
else:
    print(f"\n共删除 {total_duplicates} 条重复记录")

步骤5：重复值处理
✅ 所有股票数据均无重复值


### 2.6 步骤6：离群值标注

计算日收益率，将日收益率超过±20%的标注为 `is_extreme=True`。

**阈值选择依据**：A股有±10%涨跌停限制（ST股票为±5%），但后复权数据会反映除权除息的影响，因此可能出现超过±10%的收益率。±20%的阈值可以捕获因除权除息或特殊事件导致的极端波动。

In [8]:
print("=" * 70)
print("步骤6：离群值标注")
print("=" * 70)

extreme_summary = []
for code, df in stock_data.items():
    # 计算日收益率（基于后复权收盘价）
    df['daily_return'] = df['close'].pct_change()
    
    # 标注离群值
    df['is_extreme'] = df['daily_return'].apply(
        lambda x: True if pd.notna(x) and abs(x) > 0.20 else False
    )
    
    extreme_count = df['is_extreme'].sum()
    extreme_pct = extreme_count / len(df) * 100
    
    stock_data[code] = df
    extreme_summary.append({
        'code': code,
        'name': stock_list[code],
        'extreme_count': extreme_count,
        'extreme_pct': round(extreme_pct, 2)
    })
    
    if extreme_count > 0:
        extreme_dates = df[df['is_extreme']].index.strftime('%Y-%m-%d').tolist()
        print(f"{stock_list[code]}({code}): {extreme_count} 个极端值 ({extreme_pct:.2f}%)")
        print(f"  极端值日期: {extreme_dates[:5]}{'...' if len(extreme_dates) > 5 else ''}")

df_extreme = pd.DataFrame(extreme_summary)
print(f"\n极端值统计汇总：")
print(df_extreme.to_string(index=False))

print("\n极端值成因分析：")
print("- 后复权数据在除权除息日会出现价格跳变，导致日收益率异常")
print("- 重大资产重组、停牌复牌等事件也会导致极端波动")
print("- 这些极端值并非数据错误，而是真实市场事件，标注但不删除")

步骤6：离群值标注

极端值统计汇总：
code  extreme_count  extreme_pct  name
603063              0          0.0  禾望电气
002518              0          0.0   科士达
603118              0          0.0  共进股份
000063              0          0.0  中兴通讯
600036              0          0.0  招商银行
002142              0          0.0  宁波银行
002594              0          0.0   比亚迪
000002              0          0.0   万科A
600519              0          0.0  贵州茅台
002352              0          0.0  顺丰控股

极端值成因分析：
- 后复权数据在除权除息日会出现价格跳变，导致日收益率异常
- 重大资产重组、停牌复牌等事件也会导致极端波动
- 这些极端值并非数据错误，而是真实市场事件，标注但不删除


## 3. 宽表与长表转换

### 3.1 构建宽表

将10只股票的收盘价合并为宽表，日期为索引，每列一只股票。

In [9]:
# 构建宽表：日期为索引，每列一只股票的收盘价
close_wide = pd.DataFrame()
for code, df in stock_data.items():
    close_wide[code] = df['close']

close_wide = close_wide.dropna(how='all')  # 删除全为空的行

print("宽表（收盘价）样例：")
print(close_wide.head())
print(f"\n宽表形状: {close_wide.shape}")
print(f"列名（股票代码）: {list(close_wide.columns)}")

宽表（收盘价）样例：
            603063  002518  603118  000063  600036  002142  002594   000002  \
date                                                                          
2020-01-02    9.58   58.91   27.35  602.59  193.11   59.83   49.09  4497.51   
2020-01-03    9.55   60.10   27.49  621.29  195.69   60.03   48.96  4427.07   
2020-01-06    9.52   61.30   28.61  623.84  194.90   60.18   49.20  4352.48   
2020-01-07    9.84   64.61   29.96  622.14  194.45   60.46   48.97  4387.01   
2020-01-08    9.77   67.05   28.82  607.01  190.77   59.42   48.19  4375.96   

             600519  002352  
date                         
2020-01-02  8495.30  129.54  
2020-01-03  8108.58  128.53  
2020-01-06  8104.29  127.36  
2020-01-07  8228.64  130.30  
2020-01-08  8180.60  128.74  

宽表形状: (1540, 10)
列名（股票代码）: ['603063', '002518', '603118', '000063', '600036', '002142', '002594', '000002', '600519', '002352']


### 3.2 宽表转长表

使用 `pd.melt` 将宽表转换为长表，字段为 `date, code, close`。

In [10]:
# 宽表转长表
close_long = close_wide.reset_index().melt(
    id_vars='date',
    var_name='code',
    value_name='close'
)

print("长表（收盘价）样例：")
print(close_long.head(15))
print(f"\n长表形状: {close_long.shape}")

长表（收盘价）样例：
         date    code  close
0  2020-01-02  603063   9.58
1  2020-01-03  603063   9.55
2  2020-01-06  603063   9.52
3  2020-01-07  603063   9.84
4  2020-01-08  603063   9.77
5  2020-01-09  603063  10.14
6  2020-01-10  603063   9.95
7  2020-01-13  603063   9.99
8  2020-01-14  603063  10.01
9  2020-01-15  603063   9.83
10 2020-01-16  603063   9.99
11 2020-01-17  603063  10.15
12 2020-01-20  603063  10.00
13 2020-01-21  603063   9.81
14 2020-01-22  603063   9.81

长表形状: (15400, 3)


### 3.3 宽表与长表的适用场景

**宽表适合的操作**：
- 计算股票间的相关系数矩阵（直接调用 `.corr()`）
- 多股票走势对比可视化（每列一条线）
- PCA/因子分析（每列是一个特征维度）
- 向量化运算（如计算组合收益率）

**长表适合的操作**：
- 分组统计（`groupby` 按股票代码或行业聚合）
- ggplot/seaborn 分面绘图（`hue='code'` 或 `col='code'`）
- 存储效率高（当数据稀疏时，长表不存储空值）
- 合并其他维度的数据（如加入行业、财务指标等）

## 4. 多表合并

### 4.1 个股日度数据与指数日度数据按日期 left join

将每只股票的日度数据与沪深300日度数据按日期进行左连接。

In [11]:
print("=" * 70)
print("4.1 个股与指数日度数据合并")
print("=" * 70)

# 准备指数数据（仅保留收盘价）
hs300_close = hs300[['close']].rename(columns={'close': 'hs300_close'})

merged_data = {}
for code, df in stock_data.items():
    rows_before = len(df)
    
    # left join on date index
    df_merged = df.join(hs300_close, how='left')
    
    rows_after = len(df_merged)
    stock_data[code] = df_merged
    
    print(f"{stock_list[code]}({code}): {rows_before} → {rows_after} 行 (差: {rows_after - rows_before})")

print("\n✅ 个股与指数合并完成")

4.1 个股与指数日度数据合并
禾望电气(603063): 1540 → 1540 行 (差: 0)
科士达(002518): 1540 → 1540 行 (差: 0)
共进股份(603118): 1535 → 1535 行 (差: 0)
中兴通讯(000063): 1539 → 1539 行 (差: 0)
招商银行(600036): 1540 → 1540 行 (差: 0)
宁波银行(002142): 1534 → 1534 行 (差: 0)
比亚迪(002594): 1540 → 1540 行 (差: 0)
万科A(000002): 1540 → 1540 行 (差: 0)
贵州茅台(600519): 1540 → 1540 行 (差: 0)
顺丰控股(002352): 1537 → 1537 行 (差: 0)

✅ 个股与指数合并完成


### 4.2 月度宏观数据与日度数据合并

CPI和M2为月度数据，需要处理频率不一致问题。采用**月末对齐**策略：
- 将月度宏观数据的日期映射到所属月份的最后一天
- 与日度数据按日期left join
- 缺失的日度行向前填充宏观指标值（即同月内使用相同的宏观值）

In [12]:
print("=" * 70)
print("4.2 月度宏观数据与日度数据合并")
print("=" * 70)

# 处理CPI数据
cpi['date'] = pd.to_datetime(cpi['date'])
# 映射到月末
cpi['month_end'] = cpi['date'] + pd.offsets.MonthEnd(0)
cpi = cpi.set_index('month_end')[['cpi_yoy']]
print(f"CPI数据: {len(cpi)} 条，时间范围 {cpi.index.min()} ~ {cpi.index.max()}")

# 处理M2数据
m2['date'] = pd.to_datetime(m2['date'])
m2['month_end'] = m2['date'] + pd.offsets.MonthEnd(0)
m2 = m2.set_index('month_end')[['m2_yoy']]
print(f"M2数据: {len(m2)} 条，时间范围 {m2.index.min()} ~ {m2.index.max()}")

# 合并宏观数据到每只股票
for code, df in stock_data.items():
    rows_before = len(df)
    
    # 添加月份标记用于合并
    df['month_end'] = df.index + pd.offsets.MonthEnd(0)
    
    # 与CPI合并
    df = df.join(cpi, on='month_end', how='left')
    
    # 与M2合并
    df = df.join(m2, on='month_end', how='left')
    
    # 向前填充宏观数据（同月内使用相同值）
    df['cpi_yoy'] = df['cpi_yoy'].ffill()
    df['m2_yoy'] = df['m2_yoy'].ffill()
    
    # 删除临时列
    df = df.drop(columns=['month_end'])
    
    rows_after = len(df)
    stock_data[code] = df
    
    print(f"{stock_list[code]}({code}): {rows_before} → {rows_after} 行 (差: {rows_after - rows_before})")

print("\n✅ 宏观数据合并完成")

4.2 月度宏观数据与日度数据合并
CPI数据: 70 条，时间范围 2020-01-31 00:00:00 ~ 2025-09-30 00:00:00
M2数据: 130 条，时间范围 2020-01-31 00:00:00 ~ 2025-09-30 00:00:00
禾望电气(603063): 1540 → 2803 行 (差: 1263)
科士达(002518): 1540 → 2803 行 (差: 1263)
共进股份(603118): 1535 → 2798 行 (差: 1263)
中兴通讯(000063): 1539 → 2801 行 (差: 1262)
招商银行(600036): 1540 → 2803 行 (差: 1263)
宁波银行(002142): 1534 → 2786 行 (差: 1252)
比亚迪(002594): 1540 → 2803 行 (差: 1263)
万科A(000002): 1540 → 2803 行 (差: 1263)
贵州茅台(600519): 1540 → 2803 行 (差: 1263)
顺丰控股(002352): 1537 → 2800 行 (差: 1263)

✅ 宏观数据合并完成


## 5. 保存清洗后数据

In [13]:
# 保存每只股票清洗后的数据
for code, df in stock_data.items():
    file_path = f'data/clean/clean_{code}.csv'
    df.to_csv(file_path, encoding='utf-8-sig')
    print(f"已保存: {file_path} ({df.shape})")

已保存: data/clean/clean_603063.csv ((2803, 14))
已保存: data/clean/clean_002518.csv ((2803, 14))
已保存: data/clean/clean_603118.csv ((2798, 14))
已保存: data/clean/clean_000063.csv ((2801, 14))
已保存: data/clean/clean_600036.csv ((2803, 14))
已保存: data/clean/clean_002142.csv ((2786, 14))
已保存: data/clean/clean_002594.csv ((2803, 14))
已保存: data/clean/clean_000002.csv ((2803, 14))
已保存: data/clean/clean_600519.csv ((2803, 14))
已保存: data/clean/clean_002352.csv ((2800, 14))


In [14]:
# 构建合并数据集
all_stocks = []
for code, df in stock_data.items():
    df_copy = df.copy()
    df_copy['date'] = df_copy.index
    all_stocks.append(df_copy)

combined = pd.concat(all_stocks, ignore_index=True)
print(f"合并数据集形状: {combined.shape}")
print(f"股票数量: {combined['code'].nunique()}")
print(f"\n合并数据样例：")
print(combined.head(10))

# 保存为CSV
combined.to_csv('data/combined/combined_data.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 已保存: data/combined/combined_data.csv")

合并数据集形状: (28003, 15)
股票数量: 10

合并数据样例：
    open   high   low  close     volume      amount    code  name industry  \
0   9.49   9.64  9.45   9.58  4058396.0  38312554.0  603063  禾望电气       能源   
1   9.49   9.64  9.45   9.58  4058396.0  38312554.0  603063  禾望电气       能源   
2   9.49   9.64  9.45   9.58  4058396.0  38312554.0  603063  禾望电气       能源   
3  10.11  10.11  9.52   9.55  3102362.0  29697123.0  603063  禾望电气       能源   
4  10.11  10.11  9.52   9.55  3102362.0  29697123.0  603063  禾望电气       能源   
5  10.11  10.11  9.52   9.55  3102362.0  29697123.0  603063  禾望电气       能源   
6   9.46   9.58  9.40   9.52  3022902.0  28366553.0  603063  禾望电气       能源   
7   9.46   9.58  9.40   9.52  3022902.0  28366553.0  603063  禾望电气       能源   
8   9.46   9.58  9.40   9.52  3022902.0  28366553.0  603063  禾望电气       能源   
9   9.97   9.97  9.59   9.84  5008869.0  48294484.0  603063  禾望电气       能源   

   daily_return  is_extreme  hs300_close  cpi_yoy  m2_yoy       date  
0           NaN       False    

## 6. 多格式存储

### 6.1 Parquet格式

In [15]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PARQUET_AVAILABLE = True
except ImportError:
    PARQUET_AVAILABLE = False
    print('⚠️ PyArrow未安装，跳过Parquet部分')

if PARQUET_AVAILABLE:
    # 保存为Parquet
    combined.to_parquet('data/combined/combined_data.parquet', index=False)

    # 读取Parquet并查看Schema
    parquet_table = pq.read_table('data/combined/combined_data.parquet')
    print('Parquet Schema:')
    print(parquet_table.schema)

    # 列式读取演示：只读取close和code列
    selected_columns = pq.read_table('data/combined/combined_data.parquet', columns=['code', 'close'])
    print(f'\n列式读取结果（仅code和close列）：')
    print(selected_columns.to_pandas().head())

    # CSV vs Parquet 对比
    import os
    csv_size = os.path.getsize('data/combined/combined_data.csv')
    parquet_size = os.path.getsize('data/combined/combined_data.parquet')

    print(f'\n📊 CSV vs Parquet 对比：')
    print(f'  CSV文件大小: {csv_size / 1024:.1f} KB')
    print(f'  Parquet文件大小: {parquet_size / 1024:.1f} KB')
    print(f'  压缩比: {parquet_size / csv_size:.2%}')
    print(f'  Parquet比CSV小 {(1 - parquet_size/csv_size)*100:.1f}%')

    print('\n✅ Parquet格式存储完成')
else:
    print('💡 如需体验Parquet格式，请安装: pip install pyarrow')


⚠️ PyArrow未安装，跳过Parquet部分
💡 如需体验Parquet格式，请安装: pip install pyarrow


### 6.2 SQLite数据库

设计3张表：
1. `stock_daily` - 股票日度行情
2. `index_daily` - 指数日度行情
3. `finance_annual` - 年度财务指标

In [16]:
import sqlite3

# 创建数据库
db_path = 'data/combined/fin_data.db'
conn = sqlite3.connect(db_path)

# 表1：股票日度行情
stock_daily = combined[['date', 'code', 'name', 'industry', 'open', 'close', 
                         'high', 'low', 'volume', 'amount', 'daily_return', 
                         'is_extreme', 'hs300_close', 'cpi_yoy', 'm2_yoy']].copy()
stock_daily.to_sql('stock_daily', conn, if_exists='replace', index=False)
print(f"stock_daily: {len(stock_daily)} 行")

# 表2：指数日度行情
index_daily = pd.concat([
    hs300.reset_index()[['date', 'close']].rename(columns={'close': 'index_close'}).assign(index_code='000300', index_name='沪深300'),
    cyb.reset_index()[['date', 'close']].rename(columns={'close': 'index_close'}).assign(index_code='399006', index_name='创业板指')
])
index_daily.to_sql('index_daily', conn, if_exists='replace', index=False)
print(f"index_daily: {len(index_daily)} 行")

# 表3：年度财务指标
finance.to_sql('finance_annual', conn, if_exists='replace', index=False)
print(f"finance_annual: {len(finance)} 行")

conn.commit()
print("\n✅ SQLite数据库创建完成")

stock_daily: 28003 行
index_daily: 3080 行
finance_annual: 40 行

✅ SQLite数据库创建完成


### 6.3 SQL查询演示

演示2条有业务含义的跨表JOIN查询。

In [17]:
# 查询1：各行业2024年ROE均值及对应股票的年化收益率
query1 = '''
SELECT 
    s.industry,
    COUNT(DISTINCT s.code) as stock_count,
    AVG(f.value) as avg_roe,
    AVG(s.daily_return) * 252 as annualized_return
FROM stock_daily s
JOIN finance_annual f ON s.code = f.code 
    AND f.indicator = 'ROE' 
    AND f.year = 2024
WHERE s.daily_return IS NOT NULL
GROUP BY s.industry
ORDER BY avg_roe DESC
'''

result1 = pd.read_sql(query1, conn)
print("查询1：各行业2024年ROE均值及年化收益率")
print(result1.to_string(index=False))

# 查询2：极端波动日的市场表现
# SQLite旧版本不支持窗口函数，使用子查询替代
query2 = '''
SELECT 
    s.date,
    s.code,
    s.name,
    s.industry,
    ROUND(s.daily_return * 100, 2) as daily_return_pct,
    ROUND(s.hs300_close, 2) as hs300_close
FROM stock_daily s
WHERE s.is_extreme = 1
ORDER BY ABS(s.daily_return) DESC
LIMIT 20
'''

result2 = pd.read_sql(query2, conn)
print("\n查询2：极端波动日的市场表现（Top 20）")
print(result2.to_string(index=False))

conn.close()
print("\n✅ SQL查询演示完成")

查询1：各行业2024年ROE均值及年化收益率
industry  stock_count  avg_roe  annualized_return
     银行            1   8.0125            0.13192

查询2：极端波动日的市场表现（Top 20）
Empty DataFrame
Columns: [date, code, name, industry, daily_return_pct, hs300_close]
Index: []

✅ SQL查询演示完成


**数据清洗完成！**

下一步：运行 `03_analysis.ipynb` 进行描述性统计、可视化和回归分析。